In [3]:
 . Kiểm tra môi trường PyTorch & CUDA
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.11.0+cpu
CUDA Available: False


In [4]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 5.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import ultralytics.nn.modules as modules
import ultralytics.nn.tasks as tasks

class SpatialRelationalCoAttention(nn.Module):
    """
    Spatial-Relational Co-Attention (SRC) Module
    Spatial-Relational Co-Attention nhẹ cho feature map P2.
    Query giữ độ phân giải P2; Key/Value được giảm không gian để tiết kiệm VRAM.
    """
    def __init__(self, c1=32, reduction=8, sr_ratio=8):
        super().__init__()
        self.c1 = c1
        self.inter_c = max(16, c1 // reduction)
        self.sr_ratio = sr_ratio

        # Query: giữ chi tiết không gian ở độ phân giải P2
        self.query_proj = nn.Sequential(
            nn.Conv2d(c1, self.inter_c, kernel_size=1, bias=False),
            nn.BatchNorm2d(self.inter_c),
            nn.SiLU()
        )

        # Key: mã hóa texture/ngữ cảnh cục bộ bằng tích chập 3x3
        self.key_proj = nn.Sequential(
            nn.Conv2d(c1, self.inter_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(self.inter_c),
            nn.SiLU()
        )

        # Value được chiếu về cùng số chiều với Query/Key
        self.value_proj = nn.Conv2d(c1, self.inter_c, kernel_size=1, bias=False)

        # Spatial reduction cho Key/Value. sr_ratio=8 an toàn hơn cho P2 160x160 trên T4.
        if sr_ratio > 1:
            self.key_reduce = nn.Sequential(
                nn.Conv2d(self.inter_c, self.inter_c, kernel_size=sr_ratio, stride=sr_ratio, bias=False),
                nn.BatchNorm2d(self.inter_c)
            )
            self.value_reduce = nn.Sequential(
                nn.Conv2d(self.inter_c, self.inter_c, kernel_size=sr_ratio, stride=sr_ratio, bias=False),
                nn.BatchNorm2d(self.inter_c)
            )
        else:
            self.key_reduce = nn.Identity()
            self.value_reduce = nn.Identity()

        # Đưa attention output từ inter_c trở lại c1 channels
        self.out_conv = nn.Sequential(
            nn.Conv2d(self.inter_c, c1, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c1),
            nn.SiLU()
        )
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        if C != self.c1:
            raise RuntimeError(f'SRC expected {self.c1} channels, but received {C}')

        # scaled_dot_product_attention dùng dạng [B, heads, tokens, dim]
        q = self.query_proj(x).flatten(2).transpose(1, 2).unsqueeze(1)

        k_map = self.key_reduce(self.key_proj(x))
        v_map = self.value_reduce(self.value_proj(x))
        k = k_map.flatten(2).transpose(1, 2).unsqueeze(1)
        v = v_map.flatten(2).transpose(1, 2).unsqueeze(1)

        # PyTorch tự chọn attention kernel tiết kiệm bộ nhớ khi GPU hỗ trợ.
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
        out = out.squeeze(1).transpose(1, 2).reshape(B, self.inter_c, H, W)

        return x + self.gamma * self.out_conv(out)

# Đăng ký class vào Ultralytics để bộ parser nhận diện tự động
setattr(modules, 'SpatialRelationalCoAttention', SpatialRelationalCoAttention)
setattr(tasks, 'SpatialRelationalCoAttention', SpatialRelationalCoAttention)
print("Đã đăng ký thành công module SpatialRelationalCoAttention vào Ultralytics!")

Đã đăng ký thành công module SpatialRelationalCoAttention vào Ultralytics!


In [17]:
yaml_p2_src_architecture = """
# YOLO26n-P2 with Spatial-Relational Co-Attention (SRC)
nc: 2          # 0: fire, 1: smoke
end2end: True  # giữ thiết lập của YOLO26 detection
reg_max: 1     # DFL bins
scale: n
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]          # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]         # 1-P2/4 (160x160)
  - [-1, 2, C3k2, [256, False, 0.25]]  # 2 (Shallow spatial P2)
  - [-1, 1, Conv, [256, 3, 2]]         # 3-P3/8 (80x80)
  - [-1, 2, C3k2, [512, False, 0.25]]  # 4
  - [-1, 1, Conv, [512, 3, 2]]         # 5-P4/16 (40x40)
  - [-1, 2, C3k2, [512, True]]         # 6
  - [-1, 1, Conv, [1024, 3, 2]]        # 7-P5/32 (20x20)
  - [-1, 2, C3k2, [1024, True]]        # 8
  - [-1, 1, SPPF, [1024, 5, 3, True]]  # 9
  - [-1, 2, C2PSA, [1024]]             # 10

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 11
  - [[-1, 6], 1, Concat, [1]]                  # 12 cat backbone P4
  - [-1, 2, C3k2, [512, True]]                 # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 14
  - [[-1, 4], 1, Concat, [1]]                  # 15 cat backbone P3
  - [-1, 2, C3k2, [256, True]]                 # 16 (P3/8-small)

  # --- Nhánh P2 đón vật thể siêu nhỏ ---
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 17 (80x80 -> 160x160)
  - [[-1, 2], 1, Concat, [1]]                  # 18 cat backbone P2 (160x160)
  - [-1, 2, C3k2, [128, True]]                 # 19 (P2 Feature Map)

  # === CHÈN MODULE SPATIAL-RELATIONAL CO-ATTENTION TẠI ĐÂY ===
  # Scale n biến 128 channels khai báo ở layer 19 thành 32 channels thực tế.
  # Custom module không được parser tự scale, nên phải truyền c1=32 trực tiếp.
  - [-1, 1, SpatialRelationalCoAttention, [32, 8, 8]] # 20 (Refined P2 with memory-safe SRC)

  # --- Đường dẫn Bottom-up ---
  - [-1, 1, Conv, [128, 3, 2]]                 # 21 (160x160 -> 80x80)
  - [[-1, 16], 1, Concat, [1]]                 # 22 cat head P3
  - [-1, 2, C3k2, [256, True]]                 # 23 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]                 # 24 (80x80 -> 40x40)
  - [[-1, 13], 1, Concat, [1]]                 # 25 cat head P4
  - [-1, 2, C3k2, [512, True]]                 # 26 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]                 # 27 (40x40 -> 20x20)
  - [[-1, 10], 1, Concat, [1]]                 # 28 cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]]     # 29 (P5/32-large)

  # 4 đầu ra phát hiện: P2(tầng 20 đã qua SRC), P3(23), P4(26), P5(29)
  - [[20, 23, 26, 29], 1, Detect, [nc]]        # 30 Detect
"""

with open('/content/yolo26n-p2-src.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_p2_src_architecture.strip())

print("Đã tạo thành công file kiến trúc: /content/yolo26n-p2-src.yaml")

Đã tạo thành công file kiến trúc: /content/yolo26n-p2-src.yaml


In [18]:
import zipfile
from pathlib import Path

# 1. Đường dẫn file zip của bạn trên Drive (hãy sửa lại tên nếu bạn đặt khác)
zip_path = Path("/content/drive/MyDrive/dataset_combined.zip")

# Nếu file zip nằm ngay ở MyDrive, lệnh này sẽ giải nén thẳng vào SSD tạm của Colab (/content)
extract_dir = Path("/content/dataset_combined")
extract_dir.mkdir(parents=True, exist_ok=True)

if zip_path.exists():
    print(f" Đang giải nén {zip_path} vào {extract_dir}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(" Giải nén hoàn tất!")
else:
    print(f" Không tìm thấy {zip_path}. Hãy kiểm tra lại tên file zip trên Google Drive của bạn!")

# 2. Tìm chính xác thư mục chứa images/train vừa giải nén ra
real_root = None
for p in extract_dir.rglob("train"):
    if p.is_dir() and (p.parent.name == "images"):
        real_root = p.parent.parent
        break

if real_root:
    print(f" Đã tìm thấy root dataset chuẩn tại: {real_root}")

    # 3. Tạo file data.yaml chuẩn tuyệt đối
    yaml_content = f"""
path: {real_root.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: fire
  1: smoke
"""
    yaml_file = real_root / "data.yaml"
    yaml_file.write_text(yaml_content.strip(), encoding="utf-8")
    print(f" ĐÃ LƯU DATA.YAML CHUẨN TẠI: {yaml_file}")
else:
    print(" Chưa tìm thấy thư mục images/train trong thư mục vừa giải nén, hãy kiểm tra lại cấu trúc bên trong file zip!")

 Đang giải nén /content/drive/MyDrive/dataset_combined.zip vào /content/dataset_combined...
 Giải nén hoàn tất!
 Đã tìm thấy root dataset chuẩn tại: /content/dataset_combined/dataset_combined
 ĐÃ LƯU DATA.YAML CHUẨN TẠI: /content/dataset_combined/dataset_combined/data.yaml


In [19]:
from ultralytics import YOLO
from pathlib import Path
import re
import torch

# 1. Khởi tạo kiến trúc mới. Việc khởi tạo đã chạy một dummy forward để tính stride;
# nếu channel/shape sai thì cell này sẽ dừng ngay.
model = YOLO('/content/yolo26n-p2-src.yaml')
src_layer = model.model.model[20]
assert isinstance(src_layer, SpatialRelationalCoAttention)
assert src_layer.c1 == 32
print('✅ Build model thành công. SRC layer 20 nhận 32 channels, sr_ratio=8.')
model.info()

# 2. TÙY CHỌN: nạp checkpoint YOLO26n-P2 baseline để fine-tune với SRC.
# Không dùng resume=True vì kiến trúc đã thay đổi. Hãy thay None bằng đường dẫn best.pt thật.
BASELINE_CKPT = None
# Ví dụ:
# BASELINE_CKPT = Path('/content/drive/MyDrive/KLTN_FireSmoke/yolo26n_p2_50e/weights/best.pt')

def load_p2_baseline_with_src_shift(yolo_model, checkpoint_path, inserted_at=20):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Không tìm thấy checkpoint: {checkpoint_path}')

    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    source_model = ckpt.get('ema') if isinstance(ckpt, dict) else None
    if source_model is None and isinstance(ckpt, dict):
        source_model = ckpt.get('model')
    if source_model is None:
        source_model = ckpt
    source_state = source_model.float().state_dict() if hasattr(source_model, 'state_dict') else source_model
    target_state = yolo_model.model.state_dict()

    transferred = {}
    skipped = []
    pattern = re.compile(r'^model\.(\d+)\.(.+)$')
    for key, value in source_state.items():
        match = pattern.match(key)
        candidates = [key]
        if match:
            old_index = int(match.group(1))
            suffix = match.group(2)
            if old_index >= inserted_at:
                candidates.insert(0, f'model.{old_index + 1}.{suffix}')

        loaded = False
        for candidate in candidates:
            if candidate in target_state and target_state[candidate].shape == value.shape:
                transferred[candidate] = value
                loaded = True
                break
        if not loaded:
            skipped.append(key)

    missing, unexpected = yolo_model.model.load_state_dict(transferred, strict=False)
    print(f'✅ Đã chuyển {len(transferred)}/{len(source_state)} tensors từ baseline.')
    print(f'   Tensors mới/không nạp: {len(missing)}; source bị bỏ qua: {len(skipped)}')
    return missing, unexpected

if BASELINE_CKPT is not None:
    load_p2_baseline_with_src_shift(model, BASELINE_CKPT)
else:
    print('ℹ️ Chưa nạp baseline checkpoint: mô hình sẽ train từ khởi tạo ngẫu nhiên.')

✅ Build model thành công. SRC layer 20 nhận 32 channels, sr_ratio=8.
YOLO26n-p2-src summary: 343 layers, 2,559,441 parameters, 2,559,441 gradients, 8.8 GFLOPs
ℹ️ Chưa nạp baseline checkpoint: mô hình sẽ train từ khởi tạo ngẫu nhiên.


In [20]:
results = model.train(
    data=str(yaml_file),
    epochs=50,
    patience=10,
    imgsz=640,
    batch=8,
    device=0,
    amp=True,
    workers=4,
    project="/content/drive/MyDrive/KLTN_FireSmoke",
    name="yolo26n_p2_src_50e_b8",
    save=True
)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_combined/dataset_combined/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo26n-p2-src.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_p2_src_50e_

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      2.25G      3.037      6.359    0.03252          5        640: 100% ━━━━━━━━━━━━ 1108/1108 4.7it/s 3:56
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 119/119 5.4it/s 22.2s
                   all       1903       4599       0.18      0.218     0.0942     0.0246

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       2/50      3.77G      2.357      3.309    0.02356          2        640: 100% ━━━━━━━━━━━━ 1108/1108 4.7it/s 3:55
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 119/119 6.3it/s 18.9s
                   all       1903       4599      0.287      0.302      0.207      0.072

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/50      3.77G      2.164      2.906    0.02043          5        640: 100% ━━━━━━━━━━━━ 1108/1108 4.7it/s 3:57
                 Class     Images  Instances

In [5]:
from ultralytics import YOLO

# Nạp model có trọng số tối ưu nhất vừa train xong
best_model = YOLO('/content/drive/MyDrive/KLTN_FireSmoke/yolo26n_p2_src_50e/weights/best.pt')

# Đánh giá trên tập test
metrics = best_model.val(
    data=str(yaml_file),
    split='test',
    imgsz=640,
    device=0
)

print(f"Test mAP50:     {metrics.box.map50:.4f}")
print(f"Test mAP50-95:  {metrics.box.map:.4f}")
print(f"Test Precision: {metrics.box.mp:.4f}")
print(f"Test Recall:    {metrics.box.mr:.4f}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/KLTN_FireSmoke/yolo26n_p2_src_50e/weights/best.pt'

In [ ]:
import pandas as pd
from pathlib import Path

path_base = Path('/content/drive/MyDrive/KLTN_FireSmoke/yolo26n_p2_50e/results.csv')
path_src  = Path('/content/drive/MyDrive/KLTN_FireSmoke/yolo26n_p2_src_50e/results.csv')

def parse_best_epoch(csv_file):
    if not csv_file.exists():
        return None
    df = pd.read_csv(csv_file)
    df.columns = [c.strip() for c in df.columns]
    best_idx = df['metrics/mAP50(B)'].idxmax()
    row = df.loc[best_idx]
    return {
        'Epoch': int(row.get('epoch', best_idx)),
        'Precision': round(row.get('metrics/precision(B)', 0), 4),
        'Recall': round(row.get('metrics/recall(B)', 0), 4),
        'mAP50': round(row.get('metrics/mAP50(B)', 0), 4),
        'mAP50-95': round(row.get('metrics/mAP50-95(B)', 0), 4)
    }

data = []
b = parse_best_epoch(path_base)
s = parse_best_epoch(path_src)

if b: data.append({'Mô hình': 'YOLO26n-P2 (Baseline)', **b})
if s: data.append({'Mô hình': 'YOLO26n-P2 + SRC (Đề xuất)', **s})

if data:
    df_res = pd.DataFrame(data)
    print("\n--- BẢNG ĐỐI CHỨNG DÙNG CHO BÁO CÁO ---")
    print(df_res.to_markdown(index=False))

In [ ]:
# Chạy dự đoán trên một vài ảnh mẫu thử nghiệm
best_model.predict(
    source='/content/dataset_combined/dataset_combined/images/test',
    save=True,
    conf=0.25,
    project='/content/drive/MyDrive/KLTN_FireSmoke',
    name='src_visual_demo'
)